# Chapitre 9 - Cours 1 : modèles additifs généralisés et backfitting

Le chapitre 9 de *The Elements of Statistical Learning* présente des méthodes qui
imposent une **structure** à la fonction de régression $f(x) = \mathbb E(Y\mid X=x)$
pour échapper au fléau de la dimension : modèles additifs, arbres, PRIM, MARS,
mélanges hiérarchiques d'experts. Ce premier notebook traite des **modèles additifs
généralisés** (section 9.1 du livre). Nous allons :

1. définir le modèle additif et ses variantes « généralisées » (fonctions de lien) ;
2. rappeler ce qu'est une **spline de lissage** et la notion de lisseur linéaire ;
3. établir et implémenter l'algorithme de **backfitting** (algorithme 9.1) ;
4. l'étendre à la régression logistique par le **local scoring** (algorithme 9.2) ;
5. reproduire l'analyse des **courriels spam** : table 9.1 et figure 9.1 du livre.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl.ch09_arbres import (SplineLisseur, backfitting, GAM, logistic_irls,
                             SPAM_NAMES)

set_style()
rng = np.random.RandomState(0)

## 1. Le modèle additif

On observe une réponse $Y$ et $p$ prédicteurs $X_1,\dots,X_p$. Le modèle linéaire
suppose $\mathbb E(Y\mid X) = \alpha + \sum_j \beta_j X_j$ : l'effet de chaque variable
est une droite. Le **modèle additif** (éq. (9.1) du livre) remplace chaque droite par une
fonction quelconque (mais régulière) :
$$\mathbb E(Y\mid X_1,\dots,X_p) = \alpha + f_1(X_1) + f_2(X_2) + \dots + f_p(X_p).$$

> **Intuition.** On garde la grande force du modèle linéaire - **l'additivité** : l'effet
> de $X_j$ se lit sur un seul graphe $x_j \mapsto f_j(x_j)$, « toutes choses égales par
> ailleurs » - tout en autorisant des effets **non linéaires** (seuils, saturations…).
> En revanche, on renonce aux interactions : l'effet de $X_1$ ne dépend pas de la valeur
> de $X_2$.

**Pourquoi échappe-t-on au fléau de la dimension ?** Estimer une fonction générale de
$p$ variables demande un nombre d'observations qui explose avec $p$ (chapitre 2). Ici on
n'estime que $p$ fonctions **d'une seule variable** : chacune peut être estimée par un
lisseur unidimensionnel, comme au chapitre 5. Le prix à payer est un risque de **mauvaise
spécification** si la vraie fonction comporte des interactions.

### Modèles additifs *généralisés*

Pour une réponse binaire $Y\in\{0,1\}$, on note $\mu(X)=\Pr(Y=1\mid X)$. La régression
logistique (section 4.4) modélise le **logit** de $\mu$ par une fonction linéaire
(éq. (9.2)) ; le modèle additif logistique (éq. (9.3)) le remplace par une somme de
fonctions :
$$\log\frac{\mu(X)}{1-\mu(X)} = \alpha + f_1(X_1)+\dots+f_p(X_p).$$

Plus généralement (éq. (9.4)), on relie la moyenne conditionnelle $\mu(X)$ au
prédicteur additif par une **fonction de lien** $g$ :
$$g\big[\mu(X)\big] = \alpha + f_1(X_1) + \dots + f_p(X_p).$$
Liens classiques :

| lien | formule | usage |
|---|---|---|
| identité | $g(\mu)=\mu$ | réponse gaussienne |
| logit | $g(\mu)=\log\frac{\mu}{1-\mu}$ | réponse binaire |
| probit | $g(\mu)=\Phi^{-1}(\mu)$ | réponse binaire ($\Phi$ : f.d.r. de $\mathcal N(0,1)$) |
| log | $g(\mu)=\log \mu$ | comptages (loi de Poisson) |

> **Remarque.** Rien n'oblige toutes les $f_j$ à être non linéaires. On peut mélanger :
> des termes linéaires $X^T\beta$, des effets de facteurs qualitatifs $\alpha_k$, des
> fonctions $f(Z)$ non paramétriques, voire des fonctions de deux variables $g(Z,W)$ ou
> une courbe différente par modalité d'un facteur. Un exemple classique est la
> décomposition d'une série temporelle (éq. (9.5)) : $Y_t = S_t + T_t + \varepsilon_t$
> (saisonnalité + tendance + bruit).

## 2. Rappel : la spline de lissage, un lisseur linéaire

Pour une seule variable, la **spline de lissage cubique** (section 5.4) est la fonction
$f$ qui minimise
$$\sum_{i=1}^N \big(y_i - f(x_i)\big)^2 + \lambda \int f''(t)^2\,dt .$$
Le premier terme mesure l'ajustement, le second la **rugosité** (une droite a une
dérivée seconde nulle, donc une pénalité nulle). Le paramètre $\lambda\ge 0$ arbitre :
$\lambda=0$ interpole, $\lambda\to\infty$ donne la droite des moindres carrés.

On montre (chapitre 5) que la solution est une spline cubique naturelle à nœuds aux
$x_i$. En écrivant $f(x)=\sum_k c_k B_k(x)$ dans une base de splines, le critère devient
quadratique en $c$ :
$$\|y - Bc\|^2 + \lambda\, c^T\Omega c, \qquad \Omega_{kl}=\int B_k''(t)B_l''(t)\,dt,$$
d'où, en annulant le gradient, $\hat c = (B^TB+\lambda\Omega)^{-1}B^Ty$ et
$$\hat{\mathbf f} = B\hat c = \underbrace{B(B^TB+\lambda\Omega)^{-1}B^T}_{S_\lambda}\,y .$$

> **À retenir.** Le vecteur des valeurs ajustées est une fonction **linéaire** de $y$ :
> $\hat{\mathbf f}=S_\lambda y$. On dit que la spline de lissage est un **lisseur
> linéaire** et $S_\lambda$ est sa **matrice de lissage** ($N\times N$). Ses **degrés de
> liberté effectifs** sont $\mathrm{df}_\lambda=\mathrm{tr}(S_\lambda)$ : c'est une mesure
> de complexité plus parlante que $\lambda$.